# Kalman Filter

## Why we multiply two Gaussians — the Bayesian intuition

Before the algebra, the question worth answering is: **why do we multiply two Gaussians in the first place?**

Imagine the robot's pose is uncertain and we have **two independent estimates** of it:

- **Estimate 1** — e.g. wheel odometry, ICP, or a motion-model prediction.
- **Estimate 2** — e.g. visual odometry, optical flow, or any other sensor.

Each estimate is a probability distribution over where the robot might be — a Gaussian with its own mean and covariance. We want to **fuse** them into a single, sharper belief.

### Bayes' rule is what justifies the multiplication

$$
\underbrace{p(x \mid z)}_{\text{posterior}} \;=\; \frac{\overbrace{p(z \mid x)}^{\text{likelihood}} \; \overbrace{p(x)}^{\text{prior}}}{\underbrace{p(z)}_{\text{evidence}}}
$$

The denominator $p(z)$ is just a normalizing constant — it doesn't depend on $x$ and doesn't change the *shape* of the posterior. So up to a constant:

$$
p(x \mid z) \;\propto\; p(z \mid x)\, p(x).
$$

That is the whole reason we multiply: **one Gaussian is the prior (what we believed), the other is the likelihood (what the new measurement says), and their product is the posterior (the fused belief).**

### Why the product is still a Gaussian

A Gaussian is $\exp(-\tfrac{1}{2}(\cdot)^2)$ — its log is quadratic. Multiplying two Gaussians adds their exponents:

$$
\exp\!\big(-a x^2\big)\,\exp\!\big(-b x^2\big) \;=\; \exp\!\big(-(a+b)x^2\big),
$$

which is still a quadratic in $x$, hence still a Gaussian (after renormalizing). The mean shifts toward whichever estimate was more confident, and the variance shrinks — we've gained information.

### Where the Kalman gain comes from

When you carry out the algebra of multiplying two 1D Gaussians $\mathcal{N}(\mu_0, \sigma_0^2)$ and $\mathcal{N}(\mu_1, \sigma_1^2)$, the fused mean and variance can be rewritten compactly using a single ratio:

$$
K \;=\; \frac{\sigma_0^2}{\sigma_0^2 + \sigma_1^2},
$$

which is exactly the **Kalman gain** — "how much should I trust the new estimate relative to the old one?" The Kalman filter's update equations aren't arbitrary; they are this product-of-Gaussians identity, lifted to vectors and matrices and wrapped around a dynamic state.

The rest of this notebook walks through the algebra: the Markov assumption, the product-of-Gaussians identity, Newton's law as a state-space model, and the predict/update equations.


## Markov property

Two conditional-independence assumptions that make the filter tractable:

$$
P(Z_t \mid X_{0:t}, Z_{1:t-1}, U_{1:t}) \;=\; P(Z_t \mid X_t)
$$

$$
P(X_t \mid X_{1:t-1}, Z_{1:t-1}, U_{1:t}) \;=\; P(X_t \mid X_{t-1}, U_t)
$$

- The current measurement $Z_t$ depends only on the current state $X_t$.
- The current state $X_t$ depends only on the previous state $X_{t-1}$ and the current control input $U_t$.

## Properties of covariance

For a random vector $x$ with covariance $\Sigma$ and a deterministic matrix $A$:

$$
\operatorname{Cov}(x) = \Sigma, \qquad \operatorname{Cov}(A x) = A\,\Sigma\,A^{\!\top}.
$$

## Product of two Gaussians (1D)

The product of two 1D Gaussian densities is itself (up to a constant) a Gaussian:

$$
\mathcal{N}(\mu_0, \sigma_0^2) \cdot \mathcal{N}(\mu_1, \sigma_1^2) \;\propto\; \mathcal{N}(\mu', \sigma'^{\,2}),
$$

with

$$
\mu' = \mu_0 + \frac{\sigma_0^{2}\,(\mu_1 - \mu_0)}{\sigma_0^{2} + \sigma_1^{2}}, \qquad
\sigma'^{\,2} = \sigma_0^{2} - \frac{\sigma_0^{4}}{\sigma_0^{2} + \sigma_1^{2}}.
$$

Defining the **Kalman gain**

$$
K = \frac{\sigma_0^{2}}{\sigma_0^{2} + \sigma_1^{2}},
$$

the update collapses to the familiar form:

$$
\mu' = \mu_0 + K\,(\mu_1 - \mu_0), \qquad
\sigma'^{\,2} = \sigma_0^{2} - K\,\sigma_0^{2} = (1 - K)\,\sigma_0^{2}.
$$

## Product of two Gaussians (multivariate)

The same identity in vector form, with covariance matrices $\Sigma_0, \Sigma_1$:

$$
K = \Sigma_0\,(\Sigma_0 + \Sigma_1)^{-1},
$$

$$
\vec{\mu}' = \vec{\mu}_0 + K\,(\vec{\mu}_1 - \vec{\mu}_0),
$$

$$
\Sigma' = \Sigma_0 - K\,\Sigma_0 = (I - K)\,\Sigma_0.
$$

> Note: $\Sigma'$ is **not** $\Sigma_0^{2} - K\Sigma_0^{2}$ — covariance matrices are not "squared" the way scalar variances are written. The scalar formula $\sigma'^{\,2} = \sigma_0^{2} - K\sigma_0^{2}$ already *is* in variance units; its matrix counterpart is $\Sigma' = \Sigma_0 - K\Sigma_0$.

## Newton's law of motion as a state-space model

Constant-acceleration kinematics:

$$
x_k = x_{k-1} + v_{k-1}\,\Delta t + \tfrac{1}{2}\, a\, \Delta t^{2}, \qquad
v_k = v_{k-1} + a\,\Delta t.
$$

Stacking position and velocity into the state vector $\hat{X}_k = \begin{bmatrix} x_k \\ v_k \end{bmatrix}$:

$$
\hat{X}_k \;=\;
\underbrace{\begin{bmatrix} 1 & \Delta t \\ 0 & 1 \end{bmatrix}}_{F}\,
\hat{X}_{k-1} \;+\;
\underbrace{\begin{bmatrix} \tfrac{1}{2}\Delta t^{2} \\ \Delta t \end{bmatrix}}_{B}\, a.
$$

## Prediction (time update)

Project the state and its covariance forward through the motion model, where $Q_k$ is the process-noise covariance:

$$
\hat{X}_k \;=\; F_k\,\hat{X}_{k-1} + B_k\,\vec{u}_k,
$$

$$
P_k \;=\; F_k\,P_{k-1}\,F_k^{\!\top} + Q_k.
$$

## Correction (measurement update)

The measurement model maps state to expected observation through the measurement matrix $H_k$, with measurement-noise covariance $R_k$:

- **Predicted measurement:** $\;\hat{z}_k = H_k\,\hat{X}_k$
- **Innovation covariance:** $\;S_k = H_k\,P_k\,H_k^{\!\top} + R_k$

The Kalman gain that comes out of fusing prediction and measurement is then:

$$
K_k \;=\; P_k\,H_k^{\!\top}\,\big(H_k\,P_k\,H_k^{\!\top} + R_k\big)^{-1} \;=\; P_k\,H_k^{\!\top}\,S_k^{-1}.
$$

Updated state estimate (corrected mean):

$$
\hat{X}_k' \;=\; \hat{X}_k + K_k\,\big(Z_k - H_k\,\hat{X}_k\big).
$$

Updated covariance:

$$
P_k' \;=\; P_k - K_k\,H_k\,P_k \;=\; (I - K_k\,H_k)\,P_k.
$$
